# Threshold Analysis — Value-Based Decisioning

## Why count-based metrics are wrong for fraud

Standard precision and recall count transactions. A ₹1 fraud and a ₹10,000,000 fraud both count as "1 case." This is misleading because:

- Missing a ₹10M fraud costs ₹10M
- Blocking a legitimate ₹500 transaction costs customer friction and a small amount of business disruption

The two are not equivalent. A model that catches 90% of fraud cases but misses all the high-value ones is operationally useless — it looks good on paper and fails in practice.

**The correct question is not "how many fraud transactions did we catch?" but "how much fraud value did we catch, and at what cost to legitimate business?"**

## The four metrics that matter

At any given score threshold (the cutoff above which you block a transaction):

| Metric | Business question it answers |
|---|---|
| **Fraud value caught (₹)** | How much fraud money did we stop? |
| **% of total fraud value caught** | What fraction of all fraud are we protecting against? |
| **Legitimate value blocked (₹)** | What is the cost — how much genuine business are we disrupting? |
| **Value precision** | Of every ₹1 we block, what fraction is actual fraud vs innocent customer? |

The threshold decision is choosing where you sit on the tradeoff between the last two.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
from sklearn.model_selection import train_test_split

## Step 1 — Score the holdout set

In [ ]:
_loaded = joblib.load("../model/XGBOOST_FULL.joblib")
model = _loaded['model'] if isinstance(_loaded, dict) else _loaded

data = pd.read_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")
_, data_holdout = train_test_split(data, test_size=0.3, random_state=42)

X_holdout = data_holdout.drop('isFraud', axis=1)
y_holdout = data_holdout['isFraud']

# Build the scored holdout DataFrame
scored = pd.DataFrame(index=X_holdout.index)
scored['fraud_score'] = model.predict_proba(X_holdout)[:, 1]
scored['isFraud']     = y_holdout
scored['amount']      = X_holdout['amount']

# fraud_amount = transaction amount only if it is actually fraud, else 0
# This is the "at risk" value — what the bank would lose if this transaction goes through
scored['fraud_amount'] = scored['amount'] * scored['isFraud']
scored['legit_amount'] = scored['amount'] * (1 - scored['isFraud'])

total_fraud_value = scored['fraud_amount'].sum()
total_legit_value = scored['legit_amount'].sum()

print(f"Holdout transactions : {len(scored):,}")
print(f"Fraud cases          : {scored['isFraud'].sum():,}")
print(f"Total fraud value    : ₹{total_fraud_value:,.0f}")
print(f"Total legit value    : ₹{total_legit_value:,.0f}")

## Step 2 — Value-based threshold table

For every threshold from 0.1 to 0.99 we calculate:
- How much fraud value is blocked (caught)
- How much legitimate value is also blocked (collateral disruption)
- Value precision — what fraction of blocked value is actual fraud

In [ ]:
thresholds = np.arange(0.1, 1.00, 0.05)
rows = []

for t in thresholds:
    blocked = scored[scored['fraud_score'] >= t]

    fraud_value_caught  = blocked['fraud_amount'].sum()
    legit_value_blocked = blocked['legit_amount'].sum()
    total_value_blocked = fraud_value_caught + legit_value_blocked

    n_blocked     = len(blocked)
    n_fraud_caught = blocked['isFraud'].sum()

    # Value precision: of every ₹1 blocked, what fraction is actual fraud?
    value_precision = fraud_value_caught / total_value_blocked if total_value_blocked > 0 else 0

    # Value recall: what % of total fraud value did we catch?
    value_recall = fraud_value_caught / total_fraud_value if total_fraud_value > 0 else 0

    rows.append({
        'threshold'          : round(t, 2),
        'txns_blocked'       : n_blocked,
        'fraud_cases_caught' : int(n_fraud_caught),
        'fraud_value_caught' : fraud_value_caught,
        'legit_value_blocked': legit_value_blocked,
        'value_precision_pct': round(value_precision * 100, 2),
        'value_recall_pct'   : round(value_recall * 100, 2),
    })

results = pd.DataFrame(rows)

# Format for display
display_df = results.copy()
display_df['fraud_value_caught']  = display_df['fraud_value_caught'].apply(lambda x: f"₹{x:,.0f}")
display_df['legit_value_blocked'] = display_df['legit_value_blocked'].apply(lambda x: f"₹{x:,.0f}")
display_df.columns = ['Threshold', 'Txns Blocked', 'Fraud Cases',
                      'Fraud Value Caught', 'Legit Value Blocked',
                      'Value Precision %', 'Value Recall %']
print(display_df.to_string(index=False))

## Step 3 — Value recall vs legitimate disruption curve

This is the key chart. X-axis is legitimate value blocked (the cost). Y-axis is fraud value caught (the benefit). Every point on the curve is one threshold.

A good model hugs the top-left — catching most fraud value while disrupting very little legitimate value. The curve shows you where the diminishing returns begin: after a certain point, lowering the threshold catches very little additional fraud but blocks a lot more legitimate transactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: Fraud value recall vs legitimate value disruption ──────────────────
ax = axes[0]
ax.plot(
    results['legit_value_blocked'] / 1e9,
    results['value_recall_pct'],
    marker='o', color='steelblue', linewidth=2
)

# Annotate key thresholds on the curve
for _, row in results[results['threshold'].isin([0.5, 0.7, 0.8, 0.9, 0.95])].iterrows():
    ax.annotate(
        f"  t={row['threshold']}",
        xy=(row['legit_value_blocked'] / 1e9, row['value_recall_pct']),
        fontsize=8, color='dimgray'
    )

ax.set_xlabel("Legitimate value blocked (₹ Billions)", fontsize=11)
ax.set_ylabel("Fraud value caught (%)", fontsize=11)
ax.set_title("Fraud value caught vs Legitimate disruption", fontsize=12)
ax.grid(True, alpha=0.3)

# ── Right: Value precision across thresholds ─────────────────────────────────
ax2 = axes[1]
ax2.bar(
    results['threshold'].astype(str),
    results['value_precision_pct'],
    color='steelblue', alpha=0.8, width=0.6
)
ax2.axhline(y=95, color='red', linestyle='--', linewidth=1.2, label='95% precision line')
ax2.set_xlabel("Threshold", fontsize=11)
ax2.set_ylabel("Value Precision (% of blocked value that is fraud)", fontsize=10)
ax2.set_title("Value Precision at Each Threshold", fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("../model/threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4 — Recommended threshold and business justification

The recommended threshold is not a modelling decision. It is a business decision made by the risk team. The data scientist's job is to present the tradeoff clearly so the business can make an informed choice.

The standard framing in a bank or fintech:

- **Conservative (high threshold ~0.95):** Block very little legitimate value. Only the near-certain frauds are stopped. Low customer friction, but some fraud gets through.
- **Aggressive (lower threshold ~0.70):** Catch more fraud value. Some legitimate transactions get flagged. Higher friction, but stronger protection.
- **Tiered approach (most common in practice):** Auto-block above 0.95. Send to manual review between 0.70 and 0.95. Auto-approve below 0.70.

In [ ]:
# Tiered decisioning summary — what each band catches and costs
bands = [
    ('Auto-block',  0.95, 1.01),
    ('Review queue',0.70, 0.95),
    ('Auto-approve',0.00, 0.70),
]

print(f"{'Band':<15} {'Threshold':<18} {'Txns':>8} {'Fraud Cases':>12} {'Fraud Value Caught':>22} {'Legit Value Disrupted':>22}")
print("-" * 100)

for label, low, high in bands:
    band = scored[(scored['fraud_score'] >= low) & (scored['fraud_score'] < high)]
    fraud_val  = band['fraud_amount'].sum()
    legit_val  = band['legit_amount'].sum()
    fraud_cases = int(band['isFraud'].sum())
    txns = len(band)
    print(f"{label:<15} {f'{low:.2f} – {min(high,1.0):.2f}':<18} {txns:>8,} {fraud_cases:>12,} {f'₹{fraud_val:,.0f}':>22} {f'₹{legit_val:,.0f}':>22}")

print()
print(f"Total fraud value in holdout: ₹{total_fraud_value:,.0f}")
auto_block_fraud = scored[scored['fraud_score'] >= 0.95]['fraud_amount'].sum()
print(f"Auto-block alone catches    : ₹{auto_block_fraud:,.0f}  ({auto_block_fraud/total_fraud_value*100:.1f}% of all fraud value)")